In [8]:
# Path to the dataset file
DATA_ABS_PATH     = os.path.abspath("D:/tfm/data")
IMAGES_ABS_PATH   = os.path.abspath("D:/tfm/data/CBIS-DDSM")
PNG_ABS_PATH      = os.path.abspath("D:/tfm/data/CBIS-DDSM-PNG")
CBISDDSM_FIXED_SET = DATA_ABS_PATH + '/meta/CBIS-DDSM-fixed.parquet'

In [9]:
from pathlib import Path

# Force add the project root to sys.path (adjust as needed)
project_root = Path("../").resolve()  # one level up from /notebooks/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import mlflow
import torch
import torchvision
from torch.utils.data import DataLoader
from tqdm import tqdm

from datasets.cbisddsm import CBISDDSMDataset
from training.engine import evaluate_epoch
from training.focal_loss import FocalLoss
from utils.to_tensor_16b import ToFloatTensor16Bit

import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import optuna

import os
import time
import uuid

# Get model and parameters
model_name = "resnet50-reg-model-png"
version = 5

# Set here the URI from your MLFLow Tracking Server
TRACKING_URI = "http://localhost:5000"
client = mlflow.MlflowClient(tracking_uri=TRACKING_URI)
mlflow.set_tracking_uri(TRACKING_URI)

model_version = client.get_model_version(model_name, version)
run_id = model_version.run_id
run = client.get_run(run_id)
params = run.data.params

# Extract transform parameters
resize = int(params['resize'])
normalize_mean = eval(params['normalize_mean'])
normalize_std = eval(params['normalize_std'])
gamma = float(params['gamma'])
alpha = eval(params['alpha'])

# Create test transforms
test_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((resize, resize)),
    ToFloatTensor16Bit(),
    torchvision.transforms.Normalize(mean=normalize_mean, std=normalize_std)
])

# Create test dataset and loader
test_dataset = CBISDDSMDataset(
    "test_png.parquet",
    transform=test_transform,
    images_base_path=PNG_ABS_PATH,
    multi_view=False
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4
)

# Load model
model_uri = f"models:/{model_name}/{version}"
model = mlflow.pytorch.load_model(model_uri)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

# Create criterion
criterion = FocalLoss(gamma=gamma, alpha=alpha)

# Evaluate
test_acc, test_loss, test_recall, test_precision, test_f1, test_auc, _, _, test_class_metrics = evaluate_epoch(
    model, test_loader, criterion, device
)

# Print results
print(f"\n{'='*60}")
print(f"TEST RESULTS - {model_name} v{version}")
print(f"{'='*60}")
print(f"\nOverall Metrics:")
print(f"  Accuracy:  {test_acc:.4f}")
print(f"  Loss:      {test_loss:.4f}")
print(f"  Recall:    {test_recall:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  F1:        {test_f1:.4f}")
print(f"  AUC:       {test_auc:.4f}")

print(f"\nPer-Class Metrics:")
for class_name, metrics in test_class_metrics.items():
    print(f"\n  {class_name}:")
    for metric_name, value in metrics.items():
        print(f"    {metric_name}: {value:.4f}")

print(f"\n{'='*60}")

Evaluating:   0%|          | 0/11 [00:00<?, ?it/s]


TEST RESULTS - resnet50-reg-model-png v5

Overall Metrics:
  Accuracy:  0.8761
  Loss:      0.3318
  Recall:    0.8923
  Precision: 0.8675
  F1:        0.8776
  AUC:       0.9673

Per-Class Metrics:

  class_0:
    recall: 0.8290
    precision: 0.9146
    f1: 0.8697
    accuracy: 0.8890
    auc_roc: 0.9533

  class_1:
    recall: 0.9500
    precision: 0.8261
    f1: 0.8837
    accuracy: 0.9568
    auc_roc: 0.9920

  class_2:
    recall: 0.8977
    precision: 0.8618
    f1: 0.8794
    accuracy: 0.9063
    auc_roc: 0.9568



In [12]:
import mlflow
import torch
import tempfile
import os

model_name = "resnet50-reg-model-png"
version = 5

client = mlflow.MlflowClient(tracking_uri=TRACKING_URI)
mlflow.set_tracking_uri(TRACKING_URI)

model_version = client.get_model_version(model_name, version)
run_id = model_version.run_id

model_uri = f"models:/{model_name}/{version}"
model = mlflow.pytorch.load_model(model_uri)

with mlflow.start_run(run_name=f"wrap_{model_name}_v{version}") as new_run:
    mlflow.set_tag("wrapped_from_version", version)
    mlflow.set_tag("original_run_id", run_id)
    
    with tempfile.TemporaryDirectory() as tmpdir:
        temp_model_path = os.path.join(tmpdir, "temp_pytorch_model")
        mlflow.pytorch.save_model(model, temp_model_path)
        
        from inference.resnet_image_predictor import ResNetImagePredictor
        
        mlflow.pyfunc.log_model(
            artifact_path="image_predictor",
            python_model=ResNetImagePredictor(),
            artifacts={"pytorch_model": temp_model_path},
            pip_requirements=[
                f'mlflow=={mlflow.__version__}',
                f'torch=={torch.__version__}',
                f'torchvision=={torchvision.__version__}',
                'pillow', 'pydicom', 'numpy', 'pandas', 'opencv-python'
            ]
        )
    
    # Register as new version
    new_run_id = new_run.info.run_id
    model_uri = f"runs:/{new_run_id}/image_predictor"
    result = mlflow.register_model(
        model_uri=model_uri,
        name="resnet50-breast-cancer"  # New name for wrapped versions
    )
    
    print(f"\nServe with:")
    print(f'   $env:MLFLOW_TRACKING_URI="http://localhost:5000" mlflow models serve -m "models:/resnet50-breast-cancer/{result.version}" -p 5003 --env-manager=local"')
    print(f"{'='*60}")

2025/11/17 00:07:49 WARNING mlflow.utils.requirements_utils: Found torch version (2.8.0+cu129) contains a local version label (+cu129). MLflow logged a pip requirement for this package as 'torch==2.8.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2025/11/17 00:07:52 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.23.0+cu129) contains a local version label (+cu129). MLflow logged a pip requirement for this package as 'torchvision==0.23.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
C:\Users\Daniel\Documents\Personal\breast_cancer_detection\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature infer

2025/11/17 00:07:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'resnet50-breast-cancer' already exists. Creating a new version of this model...
2025/11/17 00:07:57 WARNING mlflow.tracking._model_registry.fluent: Run with id 2d11f15afdf249ec8d59173d3d862b30 has no artifacts at artifact path 'image_predictor', registering model based on models:/m-05a7b480a4a7417691f23f39d0e8dc7b instead
2025/11/17 00:07:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: resnet50-breast-cancer, version 5
Created version '5' of model 'resnet50-breast-cancer'.



Serve with:
   $env:MLFLOW_TRACKING_URI="http://localhost:5000" mlflow models serve -m "models:/resnet50-breast-cancer/5" -p 5003 --env-manager=local"
🏃 View run wrap_resnet50-reg-model-png_v5 at: http://localhost:5000/#/experiments/0/runs/2d11f15afdf249ec8d59173d3d862b30
🧪 View experiment at: http://localhost:5000/#/experiments/0
